# TruthShield 01: Feature caching (run once)
Runtime → Change runtime type → **T4 GPU**. Needs notebook 00 to have run once (dataset, weights and splits on Drive).

Goes through all 3,810 HiFi-AVDF videos **once** and saves, per video:
- the HAVIC feature and score,
- 8 face crops for the frequency branch,
- face and voice emotion embeddings,
- 4 quality numbers.

**Resumable:** each video is saved to Drive as soon as it is done. After a disconnect, run all again and it continues where it stopped.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, zipfile
ROOT = '/content/drive/MyDrive/TruthShield'
OUT = f'{ROOT}/cache/hifi'

%cd /content
if not os.path.exists('/content/HAVIC'):
    !git clone -q --depth 1 https://github.com/tuffy-studio/HAVIC /content/HAVIC
if not os.path.exists('/content/TruthShield'):
    !git clone -q https://github.com/Jawad374/TruthShield.git /content/TruthShield
!git -C /content/TruthShield pull -q
!pip install -q decord ffmpeg-python einops timm transformers
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # FaceX-Zoo pickled models
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

Mounted at /content/drive
/content
name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


## Videos to local disk (skipped if already there this session)

In [2]:
LOCAL = '/content/hifi'
os.makedirs(LOCAL, exist_ok=True)
for z in sorted(glob.glob(f'{ROOT}/data/HiFi-AVDF/*.zip')):
    eng = os.path.basename(z)[:-4]
    if not os.path.exists(f'{LOCAL}/.done_{eng}'):
        print('unzipping', eng)
        with zipfile.ZipFile(z) as f:
            f.extractall(LOCAL)
        open(f'{LOCAL}/.done_{eng}', 'w').close()
SPLIT = f'{ROOT}/splits/hifi_main_abs.csv'
import pandas as pd
s = pd.read_csv(SPLIT)
print(len(s), 'clips in split |', sum(os.path.exists(p) for p in s.abs_path), 'found on local disk')

unzipping kling2.5
unzipping pixverseV5
unzipping seedance1.0
unzipping sora2
unzipping veo3.1
unzipping wan2.5
3810 clips in split | 3810 found on local disk


## 1. Smoke test: 5 videos
Check that each line shows `havic_p=...` and not `FAILED`, and note the seconds per video.
These 5 are saved and skipped later, so nothing is wasted.

In [3]:
%cd /content/TruthShield
!python -u cache_features.py --split_csv {SPLIT} --havic_root /content/HAVIC --havic_weights {ROOT}/weights/best_ft_model_clean.pth --out_dir {OUT} --limit 5 --debug

/content/TruthShield
3810 clips in split, 3805 already cached, 5 to do
loading models ...
/content/HAVIC/evaluation/FaceX_Zoo/face_sdk/core/model_loader/face_detection/FaceDetModelLoader.py:22: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  model = torch.load(self.cfg['model_file_path'])
/usr/local/lib/python3.13/dist-packages/torch/serialization.py:1831: SourceChangeWarning: source code of class 'torchvision.models._utils.IntermediateLayerGetter' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
/usr/local/lib/python3.13/dist-packages/torch/serialization.py:1831: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has changed. you can retrieve the origi

## 2. Cache in parts: one generator per part
Every video is saved to Drive the moment it is done, and each generator is one part.
Run the status cell, then the part cell. `PARTS_THIS_SESSION` limits how many parts run now.
After a disconnect, reconnect and run again: finished videos and parts are skipped.

In [4]:
import glob
ENGINES = ['kling2.5', 'pixverseV5', 'seedance1.0', 'sora2', 'veo3.1', 'wan2.5']
def status():
    done = {os.path.basename(f) for f in glob.glob(f'{OUT}/clips/*.pt')}
    rows = []
    for e in ENGINES:
        sub = s[s.engine == e]
        n = sum(p.replace('/', '__').replace('.mp4', '.pt') in done for p in sub.video_path)
        rows.append((e, n, len(sub), 'DONE' if n == len(sub) else ''))
    print(pd.DataFrame(rows, columns=['part (generator)', 'cached', 'total', '']).to_string(index=False))
    print(f'TOTAL: {sum(r[1] for r in rows)} / {len(s)}')
    return [r[0] for r in rows if r[1] < r[2]]
remaining = status()

part (generator)  cached  total 
        kling2.5       5    460 
      pixverseV5       0    616 
     seedance1.0       0    788 
           sora2       0    728 
          veo3.1       0    638 
          wan2.5       0    580 
TOTAL: 5 / 3810


In [ ]:
PARTS_THIS_SESSION = 6   # e.g. 2 = only cache 2 generators now, the rest later
%cd /content/TruthShield
for e in remaining[:PARTS_THIS_SESSION]:
    print(f'========== PART: {e} ==========')
    !python -u cache_features.py --split_csv {SPLIT} --havic_root /content/HAVIC --havic_weights {ROOT}/weights/best_ft_model_clean.pth --out_dir {OUT} --engines {e}
remaining = status()

/content/TruthShield
========== PART: kling2.5 ==========
460 clips in split, 5 already cached, 455 to do
loading models ...
/content/HAVIC/evaluation/FaceX_Zoo/face_sdk/core/model_loader/face_detection/FaceDetModelLoader.py:22: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  model = torch.load(self.cfg['model_file_path'])
/usr/local/lib/python3.13/dist-packages/torch/serialization.py:1831: SourceChangeWarning: source code of class 'torchvision.models._utils.IntermediateLayerGetter' has changed. you can retrieve the original source code by accessing the object's source attribute or set `torch.nn.Module.dump_patches = True` and use the patch tool to revert the changes.
  _check_container_source(*data)
/usr/local/lib/python3.13/dist-packages/torch/serialization.py:1831: SourceChangeWarning: source code of class 'torch.nn.modules.container.Sequential' has 

## 3. Merge all parts into 2 files for fast training (run when every part is DONE)

In [ ]:
n = len(glob.glob(f'{OUT}/clips/*.pt'))
print(f'{n} / {len(s)} clips cached')
if os.path.exists(f'{OUT}/failures.csv'):
    f = pd.read_csv(f'{OUT}/failures.csv').drop_duplicates('video_path')
    f = f[~f.video_path.map(lambda p: os.path.exists(f"{OUT}/clips/{p.replace('/', '__').replace('.mp4', '.pt')}"))]
    print(len(f), 'clips failed:'); print(f.error.str.slice(0, 60).value_counts().head(10))

In [ ]:
%cd /content/TruthShield
!python -u cache_features.py --pack --out_dir {OUT}
!ls -lh {OUT}/features.pt {OUT}/faces.npy